# Flet Declarative UI

Recall that in the imperative approach we had to tell the framework exactly how to build and update the interface step by step. That is, we manipulate the UI directly by creating controls, changing their properties, and inserting or removing them in response to user actions (or app state). This is still a valid approach especially for small applications.

The problem with the imperative approach is that the state, logic, and building UI elements all live in the same place. Suppose you have $(s_1, \ldots, s_n)$ **state variables**[^state_vars_not_encouraged]. All components that depend on each of these have to respond to changes in the variables. Since the app is build imperatively, the changes in the components in response to the change have to be hard-coded and reasoned individually reasoned with by the developer. This task grows exponentially as $n$ increases. You have seen a glimpse of this in [the previous notebook](/topics/apps/02-flet.html). 

From the [Flet blog post](https://flet.dev/blog/introducing-declarative-ui-in-flet):

> Dogfooding Flet — building our own products like the Flet mobile app and the Control Gallery — made it clear that the imperative approach becomes hard to manage as apps grow. That's why **Flet 1.0** introduces a declarative approach alongside the existing imperative API, drawing inspiration from frameworks such as React, SwiftUI, and Jetpack Compose.

[^state_vars_not_encouraged]: Actually, using state variables to ground the application state is not even encouraged natively by the imperative approach. You can just build the application any way you like &mdash; which, incidentally, is also its strength.

## What is declarative UI? 

The declarative approach means you describe [*what*]{.underline} the UI should look like for a given state instead of [*how*]{.underline} to build the UI. In a sense, it's implicit vs explicit programming. As such, the declarative approach will feel like magic and go over your head when you start coding with it (coming from someone with zero frontend experience). The framework figures out the minimal updates needed to reflect the change in UI w.r.t. change in state so it always stays [consistent across renders]{.mark}. 

:::{.callout-note}
We like to write this in the following slogan: $\text{UI} = f(\text{state}).$ Our goal is to make the code simpler, more predictable, and easier to reason about.
:::

## Hello, world! (declarative)

Let's reproduce [our previous](/topics/apps/01-flet.html#hello-world) "Hello, world!" program using the declarative approach:

```{.python filename="src/hello.py"}
import flet as ft
import asyncio
from random import randint


hello_world = [
    "Hello, world!",
    "¡Hola, mundo!",
    "Bonjour, monde !",
    "Hallo, Welt!",
    "Ciao, mondo!",
    "Olá, mundo!",
    "こんにちは、世界！",
    "안녕하세요, 세계!",
    "你好，世界！",
    "مرحباً، يا عالم!",
]

@ft.component
def Greeting(greeting: str) -> ft.Container:
    return ft.Container(
        ft.Text(greeting, size=60),
        alignment=ft.Alignment.CENTER,
        expand=True
    )

@ft.component
def RollButton(on_click):
    return ft.FloatingActionButton(
        content=ft.Icon(ft.Icons.CASINO, size=60),
        on_click=on_click,
        height=60, width=60
    )


@ft.component
def AppView() -> ft.Column:
    n = len(hello_world)
    greeting, set_greeting = ft.use_state(hello_world[0])
    
    async def roll_greeting(e):
        set_greeting("")
        await asyncio.sleep(0.2)
        set_greeting(hello_world[randint(0, n - 1)])
    
    return ft.Column(
        controls=[
            Greeting(greeting),
            ft.Row(
                controls=[RollButton(roll_greeting)],
                alignment=ft.MainAxisAlignment.END
            ),
        ],
        alignment=ft.Alignment.CENTER,
        expand=True
    )


if __name__ == "__main__":
    ft.run(lambda page: page.render(AppView))
```

<video
  src="./img/flet-declarative/hello-world.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

This behaves exactly as before. The critical difference is this part of the code:

```python
greeting, set_greeting = ft.use_state(hello_world[0])

async def roll_greeting(e):
    set_greeting("")
    await asyncio.sleep(0.2)
    set_greeting(hello_world[randint(0, n - 1)])
```

... and that we're rendering with `ft.render` the function: 

```python
@ft.component
def AppView() -> ft.Control:
    ...
```

which returns a Flet control. This function returns the UI every time the app's state `greeting` changes. The state variable is defined using the `ft.use_state`. We will explain this function shortly, for now think of `greeting` as the state variable that the component monitors for a change, and `set_greeting` (`str -> None`) as a setter. Notice that we aren't making any explicit page updates. The UI automatically refreshes when the value of `greeting` changes.

Also notice that the hook is now asynchronous. This is necessary since the assignment of the value of `use_state` variable is scheduled in the async event loop, along with the UI refresh. For example, if we don't put `asyncio.sleep` here, then the transition into empty text will not register in the UI since control is not [yielded]{.mark} to the Flet framework in between the state variable updates.

## Declarative elements

### Components

A **component** is simply a [reusable function]{.underline} that returns a control as a function of the state. It should be a self-contained unit of UI that takes inputs (properties, data, event handlers) and returns Flet controls. Every time its inputs or internal state change, the component rebuilds its UI, and Flet automatically updates only the changed parts. In the above example:

```python
@ft.component
def Greeting(text: str) -> ft.Container:
    return ft.Container(
        ft.Text(text, size=60),
        alignment=ft.Alignment.CENTER,
        expand=True
    )

@ft.component
def RollButton(on_click):
    return ft.Row(
        controls=[
            ft.FloatingActionButton(
                content=ft.Icon(ft.Icons.CASINO, size=60),
                on_click=on_click,
                height=60, width=60
            )
        ],
        alignment=ft.MainAxisAlignment.END
    )
```

The first one returns a container for a text greeting that is centered, while the other is a row containing the stylized button aligned to the right.

### Hooks, state variables

Hooks are lightweight function (i.e. they have to react to events instantaneously) that let components **store state**, **react to lifecycle events**, or **access shared context**. Moreover, these are all accomplished without writing classes or managing manual state objects. In our example, we have `roll_greeting` as a hook that sets the `greeting` variable. Note that when `set_greeting` is called Flet re-runs the component and re-renders only what changed (i.e. the output of the `Greeting` component).

The `use_state` function gives this variable a **persistent state**. That persistence is crucial: ordinary local variables are re-created on every render, so their values would disappear. Hook state survives re-renders, giving your functional components memory, and allowing shared data, without resorting to globals or classes.

:::{.callout-tip}
Flet offers the following built-in hooks: 

| | |
| :------- | :--- |
| [`use_state`](https://docs.flet.dev/types/use_state/) | Store local state across rebuilds. |
| [`use_effect`](https://docs.flet.dev/types/use_effect/) | Run side effects when something changes. | 
| [`use_context`](https://docs.flet.dev/types/use_context/)  | Access shared data or services. | 
| [`use_memo`](https://docs.flet.dev/types/use_memo/) | Memoize computed values. |
| [`use_ref`](https://docs.flet.dev/types/use_ref/) | Preserve a mutable value for the lifetime of the component without causing re-renders. |

:::

### Observables

One can think of `use_state` variables as **local**, **component-scoped** persistent state. Meanwhile, we have application state that is, in a sense, global and reflects a truth. In the declarative approach, we can represent application state in terms of **observables**. An observable is a [reactive]{.mark} data holder that keeps UI in sync automatically. That is, whenever the values of observable change, the corresponding parts of the UI that depend on it are notifed and update instantly and efficiently. 


Observables can be created in two ways:

```python
@dataclass
class CounterState(ft.Observable):
    count: int
```

Or using a decorator:

```python
@dataclass
@ft.observable
class CounterState:
    count: int
```

Observables fit nicely into Flet's declarative approach in that a component that depends on an **observable parameter** (see below example to see how this is implemented) automatically re-renders when that observable updates. Moreover, hooks that reference observables trigger a re-render when the observable changes. For example, we may see examples with:

```python
@ft.component
def AppView() -> ft.Column:
    todo, _ = ft.use_state(AppState())
```

where `AppState` is an observable. This is done in the ff. example:


```{.python filename=src/counter.py}
import asyncio
from dataclasses import dataclass

import flet as ft

@dataclass
@ft.observable
class AppState:
    counter: float

    async def start_counter(self):
        self.counter = 0                # <1>
        for _ in range(0, 10):
            self.counter += 0.1                 # <2>
            await asyncio.sleep(0.5)


@ft.component
def AppView():
    state, _ = ft.use_state(AppState(counter=0))    # <3>

    return [
        ft.ProgressBar(state.counter),      # <4>
        ft.Button("Run!", on_click=state.start_counter),    # <5>
    ]

ft.run(lambda page: page.render(AppView))
```

1. Updating an observable to `0` using `=`. Flet detects a change ⇒ re-renders UI.
2. Incrementing an observable. This is a common gotcha. For immutable types (`int`, `str`, etc) `x += a` is the same as `x = x + a`. But for mutable types this is an in-place operation and Flet will **not** detect a change. ⚠️ Here it's fine.
3. Here an instance of the `AppState` is used as state variable to trigger UI re-render. 
4. An observable parameter is assigned to the `ProgressBar`. 
5. Observables can define **methods** of modifying its internal attributes based on events, which we can use as hooks.

:::{.callout-tip}
The distinction between **app state** (observables, "truth") and **app view** (UI components, "rendered") is a good mental model when designing declarative applications.
:::

<video
  src="./img/flet-declarative/counter.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

:::{.callout-caution}
In Flet’s declarative model, an `Observable` triggers UI updates when the **observable reference itself is reassigned**, not when the underlying object is mutated in place. Operations like `=` create a new value and rebind the observable, which Flet detects and use as a signal to re-render dependent components. 

The behavior of `+=` depends on the type: for **immutable types** (`int`, `str`, `tuple`), `+=` behaves like `=` and triggers a re-render, but for **mutable types** (`list`, `dict`, `set`), `+=` mutates the object in place and does **not** trigger a re-render. Direct mutations (e.g., `list.append()` or `dict["k"] = v`) similarly leave the observable reference unchanged and do not refresh the UI. 

This design mirrors many reactive frameworks where reactivity is tied to assignment semantics rather than deep mutation tracking, keeping the system simpler and more predictable, but requiring explicit reassignment (e.g., `obs.value = obs.value.copy()` or `obs.value = obs.value + [x]`) when working with mutable structures.
:::

## Example: Declarative Todo App